
Installare 

Tesseract

OpenCV

python -m pip install ot
py -m pip install opencv-contrib-python

In [ ]:

import os
import cv2
import numpy as np
import pytesseract
from pdf2image import convert_from_path
import json

#PDF_FOLDER= f"data/pdf/"
#TXT_FOLDER= f"data/txt2/"


In [ ]:
def process_pdfs_in_folder(pdf_folder, txt_folder, tesseract_cmd):
    # esegue Tesseract
    pytesseract.pytesseract.tesseract_cmd = tesseract_cmd    
    # Scansiona i file nella cartella
    for filename in os.listdir(pdf_folder):
        if filename.lower().endswith(".pdf"):
            pdf_path_filename = os.path.join(pdf_folder, filename)
            print(f"esecuzione: {pdf_path_filename}")            
            # Converti il PDF in immagini
            images = convert_from_path(pdf_path_filename)            
            # Estrai il testo da ogni immagine
            extracted_text = ""
            for i, image in enumerate(images):
                text = pytesseract.image_to_string(image)
                extracted_text += text + "\n"
                print(f"pagina {i+1} ...")            
            # Salva il testo estratto in un file di output
            txt_filename = os.path.splitext(filename)[0] + ".txt"
            output_text_path = os.path.join(txt_folder, txt_filename)
            with open(output_text_path, "w", encoding="utf-8") as text_file:
                text_file.write(extracted_text)
            print(f"Salva il testo estratto nel file: {output_text_path}\n")


 
tesseract_cmd='C:/Program Files/Tesseract-OCR/tesseract.exe'
process_pdfs_in_folder(PDF_FOLDER, TXT_FOLDER, tesseract_cmd)

In [ ]:



def process_pdfs_in_folder(
    pdf_folder,
    txt_folder,
    tesseract_cmd,
    poppler_path,
    skip_existing=True,
    save_json=False
):

    pytesseract.pytesseract.tesseract_cmd = tesseract_cmd

    os.makedirs(txt_folder, exist_ok=True)

    for filename in os.listdir(pdf_folder):

        if not filename.lower().endswith(".pdf"):
            continue

        pdf_path = os.path.join(pdf_folder, filename)
        txt_filename = os.path.splitext(filename)[0] + ".txt"
        txt_path = os.path.join(txt_folder, txt_filename)

        # SKIP opzionale
        if skip_existing and os.path.exists(txt_path):
            print(f"⏭️ skip: {filename}")
            continue

        print(f"\n📄 elaboro: {filename}")

        try:

            images = convert_from_path(
                pdf_path,
                dpi=300,
                poppler_path=poppler_path
            )

            extracted_text = ""
            pages = []

            for i, image in enumerate(images):

                print(f"pagina {i+1}")

                img = cv2.cvtColor(np.array(image), cv2.COLOR_BGR2GRAY)
                img = cv2.threshold(
                    img, 0, 255,
                    cv2.THRESH_BINARY + cv2.THRESH_OTSU
                )[1]

                text = pytesseract.image_to_string(
                    img,
                    lang="ita",
                    config="--psm 6"
                )

                extracted_text += text + "\n"

                if save_json:
                    pages.append({
                        "page": i + 1,
                        "text": text
                    })

            # salva TXT
            with open(txt_path, "w", encoding="utf-8") as f:
                f.write(extracted_text)

            print(f"✅ salvato TXT: {txt_path}")

            # salva JSON opzionale
            if save_json:

                json_filename = os.path.splitext(filename)[0] + ".json"
                json_path = os.path.join(txt_folder, json_filename)

                with open(json_path, "w", encoding="utf-8") as f:
                    json.dump({
                        "file": filename,
                        "pages": pages
                    }, f, indent=2, ensure_ascii=False)

                print(f"✅ salvato JSON: {json_path}")

        except Exception as e:

            print(f"❌ errore su {filename}")
            print(e)

In [ ]:
PDF_FOLDER= r"prova/"
TXT_FOLDER= r"prova/"

TESSERACT = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
POPPLER = r"C:\Program Files\poppler\poppler-25.12.0\Library\bin"

process_pdfs_in_folder(
    PDF_FOLDER,
    TXT_FOLDER,
    TESSERACT,
    POPPLER,
    skip_existing=True,   # skip se TXT già esiste
    save_json=False       # JSON opzionale
)

In [33]:
import csv
import os
import pandas as pd
from pathlib import Path

files = [
    ("classification/ELENCO COMUNITA DI MONTAGNA - COMUNI PER RIGA.CSV", "Comunità di montagna"),
    ("classification/ELENCO COMUNITA TERRITORIALI DEL TRENTINO - COMUNI PER RIGA.CSV", "Comunità territoriale"),
    ("classification/ELENCO UNIONI DI COMUNI - COMUNI PER RIGA.CSV", "Unione di comuni"),
]

PROVINCE_MAP = {
    "TN": "Trento",
    "IM": "Imperia",
    "GE": "Genova",
    "SV": "Savona",
}

def normalizza_provincia(provincia):
    if pd.isna(provincia):
        return provincia

    p = str(provincia).strip()
    p_upper = p.upper()

    if p_upper in PROVINCE_MAP:
        return PROVINCE_MAP[p_upper]

    return p

def parse_line_to_7_fields(row):
    """
    Rebuild malformed rows.
    Target schema:
    COD, DENOMINAZIONE, REGIONE, PROV., COMUNE, TOT. POPOLAZIONE, TOT. COMUNI
    """
    if len(row) == 7:
        return row
    if len(row) > 7:
        cod = row[0].strip()
        tot_comuni = row[-1].strip()
        tot_pop = row[-2].strip()
        comune = row[-3].strip()
        prov = row[-4].strip()
        regione = row[-5].strip()
        denominazione = ",".join(part.strip() for part in row[1:-5]).strip()
        return [cod, denominazione, regione, prov, comune, tot_pop, tot_comuni]
    return None

records = []

for path, tipo_ente in files:
    with open(Path(path), "r", encoding="utf-8", errors="replace", newline="") as f:
        reader = csv.reader(f)
        header = next(reader, None)

        for row in reader:
            if not row:
                continue

            row = parse_line_to_7_fields(row)
            if row is None:
                continue

            cod, denominazione, regione, provincia, comune_field, tot_pop, tot_comuni = [str(x).strip() for x in row]
            provincia = normalizza_provincia(provincia)

            comuni = [c.strip() for c in comune_field.split(",") if c.strip()]

            for comune in comuni:
                records.append({
                    "TIPO_ENTE": tipo_ente,
                    "COD": cod,
                    "DENOMINAZIONE": denominazione,
                    "COMUNE": comune,
                    "PROVINCIA": provincia,
                    "REGIONE": regione,
                })

df = pd.DataFrame(records).drop_duplicates()

df["COD_NUM"] = pd.to_numeric(df["COD"], errors="coerce")
df = df.sort_values(["TIPO_ENTE", "COD_NUM", "DENOMINAZIONE", "COMUNE"]).drop(columns=["COD_NUM"])

csv_path = Path(r"classification/cls_elenco_unioni_comuni_comunita_di_montagna_comunità_territoriali.csv")
xlsx_path = Path(r"classification//cls_elenco_unioni_comuni_comunita_di_montagna_comunità_territoriali.xlsx")

df.to_csv(csv_path, index=False, encoding="utf-8-sig")
df.to_excel(xlsx_path, index=False)

df.head(15)

,TIPO_ENTE,COD,DENOMINAZIONE,COMUNE,PROVINCIA,REGIONE
0,Comunità di montagna,1,Comunità del Collio,Capriva del Friuli,Gorizia,Friuli-Venezia Giulia
1,Comunità di montagna,1,Comunità del Collio,Cormons,Gorizia,Friuli-Venezia Giulia
2,Comunità di montagna,1,Comunità del Collio,Dolegna del Collio,Gorizia,Friuli-Venezia Giulia
3,Comunità di montagna,1,Comunità del Collio,Farra d'Isonzo,Gorizia,Friuli-Venezia Giulia
4,Comunità di montagna,1,Comunità del Collio,Mariano del Friuli,Gorizia,Friuli-Venezia Giulia
5,Comunità di montagna,1,Comunità del Collio,Medea,Gorizia,Friuli-Venezia Giulia
6,Comunità di montagna,1,Comunità del Collio,Moraro,Gorizia,Friuli-Venezia Giulia
7,Comunità di montagna,1,Comunità del Collio,Mossa,Gorizia,Friuli-Venezia Giulia
8,Comunità di montagna,1,Comunità del Collio,San Floriano del Collio-Števerjan,Gorizia,Friuli-Venezia Giulia
9,Comunità di montagna,1,Comunità del Collio,San Lorenzo Isontino,Gorizia,Friuli-Venezia Giulia
